In [9]:
import numpy as np
import pandas as pd

In [ ]:
def percentileofscore(a, score):
    a = np.asarray(a)
    n = len(a)
    score = np.asarray(score)
    # Prepare broadcasting
    score = score[..., None]
    def count(x):
        return np.count_nonzero(x, -1)
    
    left = count(a < score)
    right = count(a <= score)
    plus1 = left < right
    perct = np.ma.filled((left + right + plus1) * (50. / n), np.nan)
    return perct

In [8]:
a = [1,2,3,4,5]
score1 = 3
score2 = 2.5
pscore = percentileofscore(a, score1)
pscore2 = percentileofscore(a, score2)
print(pscore)
print(pscore2)


filled1 = np.ma.filled(np.array([np.inf, -1]), np.nan)
print(filled1)
# filled2 = np.ma.filled(0, np.nan)

60.0
40.0
[inf -1.]


In [ ]:
def norm(x, rolling_window=2000): # 20230910 checked, 不再用L2 norm，恢复到之前的zscore，然后这里需要做的是给他增加一个周期

    # x = np.log1p(np.asarray(x)) # 原有写法factor_mean.values
    # arr = np.asarray(x)
    # epsilon = 1e-8  # 小常数
    # mean_abs = np.abs(np.mean(arr))
    # x = np.sign(arr) * np.log1p(np.abs(arr)) / np.log1p(mean_abs + epsilon)
    
    factors_data = pd.DataFrame(x, columns=['factor'])
    factors_data = factors_data.replace([np.inf, -np.inf, np.nan], 0.0)      
    # factors_mean = factors_data.rolling(window=rolling_window, min_periods=1).mean()
    factors_std = factors_data.rolling(window=rolling_window, min_periods=1).std()
    factor_value = (factors_data) / factors_std
    # factor_value = (factors_data - factors_mean) / factors_std
    # factor_value = factor_value.apply(np.log1p) # 这样会导致均值明显不为零
    factor_value = factor_value.replace([np.inf, -np.inf, np.nan], 0.0) 
    # factor_value = factor_value.clip(-6, 6)
    # 最终的定稿应该是，先给他log1p再去norm，因为这样会让他的mean为0，skew为0，kurtosis为7
    return np.nan_to_num(factor_value).flatten()



In [ ]:
def _sigmoid(x1):
    """Special case of logistic function to transform to probabilities."""
    with np.errstate(over='ignore', under='ignore'):
        return norm(np.nan_to_num(1 / (1 + np.exp(-x1))))
    
